# 00 · Común: configuración, rutas y funciones compartidas

Este notebook **no se ejecuta solo**: los demás lo cargan en su primera celda con

```python
%run 00_comun.ipynb
```

Así cada función existe **una sola vez** (antes la evaluación de tracking estaba copiada en tres scripts).
Funciona con los dos kernels (`colibri-yolo` y `colibri-vda`): no importa Ultralytics ni torch al cargarse.

| Sección | Qué define | Lo usan |
|---|---|---|
| 1. Configuración | `CFG`, `P` (rutas), `MODO`, `.env` | todos |
| 2. Fuentes | `FUENTES`, `video_path`, `videos`, `corto` | todos |
| 3. Etiquetas y verdad de terreno | `yolo_boxes`, `cargar_manifiesto`, `cargar_gt_test`, `cargar_gt_union` | 01, 02, 03 |
| 4. Comparar con el sprint | `comparar` | todos |
| 5. Tracking | `tracker_cfg`, `track_video`, `evaluate_tracks`, `build_trajectory`, `get_tracks` | 03 |
| 6. Unión de fragmentos | `union_tuned`, `union_reference`, `visit_metrics`, `bridges`, `draw` | 03 |
| 7. Siete puntos | `smooth`, `uniform_time`, `visvalingam`, `catmull_rom`, `siete_puntos`... | 04, 06, herramienta |
| 8. Profundidad | `read_frame`, `motion_mask`, `make_backend`, `leer_puntos_z`, `figura_3d`, `mapas_vda`... | 05, 06, herramienta |

El código general (tracking, unión, puntos y profundidad) vive en el paquete `colibri/` de la raíz del repositorio, y aquí
queda lo propio de la investigación: los videos del proyecto, la verdad de terreno y las comparaciones con cada sprint.
Los scripts originales siguen en `notebooks/historico/scripts/`.

## 1. Configuración

Busca `config.yaml` subiendo desde la carpeta actual, así el notebook funciona desde `notebooks/` o desde la raíz.
`MODO` se puede forzar sin tocar el YAML con la variable de entorno `COLIBRI_MODO`.
La clave de Roboflow se lee de `.env` **solo a la memoria del proceso**: nunca se imprime ni se escribe.

In [ ]:
import json, os, re, sys, hashlib, warnings
from pathlib import Path, PureWindowsPath

import cv2
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from scipy.signal import savgol_filter
from IPython.display import display
from colibri.seguimiento import *
from colibri.puntos import *
from colibri.profundidad import *

def _repo_root(start=Path.cwd()):
    for d in (start, *start.parents):
        if (d / "config.yaml").exists() and (d / "notebooks").is_dir():
            return d
    raise FileNotFoundError("No encuentro config.yaml: abre el notebook desde la carpeta del repositorio")

REPO = _repo_root()
CFG = yaml.safe_load((REPO / "config.yaml").read_text(encoding="utf-8"))
if (REPO / "config.local.yaml").exists():      # ajustes de tu equipo (fuera de git), p. ej. zona_trabajo
    CFG.update(yaml.safe_load((REPO / "config.local.yaml").read_text(encoding="utf-8")))
ZONA = (REPO / CFG["zona_trabajo"]).resolve()
P = {k: (ZONA / v) for k, v in CFG["rutas"].items()}
P.update(repo=REPO, zona=ZONA, dd=REPO / "datos_derivados", resultados=REPO / "resultados", modelos_repo=REPO / "modelos")
MODO = os.environ.get("COLIBRI_MODO", CFG["modo"])
assert MODO in ("artefactos", "recalcular"), MODO
SEED = CFG["semilla"]
TOL = CFG["tolerancia"]
DETECTOR = REPO / CFG["vigente"]["detector"]
DETECTOR_ANTERIOR = P["modelos"] / CFG["vigente"]["detector_anterior"]
RUNS = P["runs"]
for k in ("runs", "frames"):
    P[k].mkdir(parents=True, exist_ok=True)

def cargar_env(path=REPO.parent / ".env"):
    """Carga KEY=valor de .env en os.environ sin mostrar nada. Devuelve los nombres cargados (no los valores)."""
    if not path.exists():
        return []
    names = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            k, v = line.split("=", 1)
            if v.strip():
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'")); names.append(k.strip())
    return names

_env = cargar_env()
print(f"repo {REPO} | zona de trabajo {ZONA} | MODO = {MODO} | .env: {', '.join(_env) or 'no hay'}")

## 2. Fuentes: los 12 videos

`datos_derivados/fuentes.csv` es el inventario: id, archivo original, carpeta (`vv`, `train`, `test`), origen y cifras
esperadas (fps y frames). Los videos **no se redistribuyen** (Adobe Stock y YouTube); el notebook 01 explica cómo
conseguirlos y comprueba que coinciden. `video_id` es el nombre que usan todos los CSV del proyecto.

In [ ]:
FUENTES = pd.read_csv(P["dd"] / "fuentes.csv")

def slug(stem, maxlen=48):
    s = re.sub(r"[^a-z0-9]+", "_", stem.lower()).strip("_")
    return s[:maxlen].rstrip("_")

def video_path(video_id) -> Path:
    r = FUENTES.set_index("video_id").loc[video_id]
    return P["videos"] / r.carpeta / r.archivo

def corto(video_id):
    """Nombre corto (DoJ1, Gu30, i5bE, J4PK, A129...) a partir del video_id o de cualquier texto que lo contenga."""
    for r in FUENTES.itertuples():
        if r.video_id in video_id or str(r.id_origen).lower() in video_id.lower():
            return r.corto
    return video_id

def videos(roles=("vv", "train", "test"), solo_ajuste_union=False, omitir_faltantes=False) -> dict:
    """Equivale a list_videos() de union.py: {video_id: dict(path, rol, W, H, fps)} leyendo cada video.
    Con omitir_faltantes=True, los videos que no estén se saltan (y se listan en FALTAN) en vez de dar error."""
    global FALTAN
    out, FALTAN = {}, []
    for r in FUENTES[FUENTES.rol.isin(roles)].itertuples():
        if solo_ajuste_union and r.rol == "train" and not r.en_ajuste_union:
            continue
        v = dict(path=video_path(r.video_id), rol=r.rol)
        cap = cv2.VideoCapture(str(v["path"]))
        if not cap.isOpened() and omitir_faltantes:
            FALTAN.append(r.corto); continue
        if not cap.isOpened():
            raise FileNotFoundError(f"Falta el video {v['path']} (ver notebook 01, sección 1)")
        v.update(W=int(cap.get(3)), H=int(cap.get(4)), fps=cap.get(cv2.CAP_PROP_FPS)); cap.release()
        out[r.video_id] = v
    return out

## 3. Etiquetas y verdad de terreno

Las etiquetas de `humming` (nuestro proyecto de Roboflow) están **congeladas en el repo**, en
`datos_derivados/humming/etiquetas_{v1,v2}/{train,valid,test}/<video_id>_fNNNNN.txt`. Los manifiestos
(`split_manifest_v1.csv`, `split_manifest_v2.csv`) conservan las rutas de Windows con las que se crearon; aquí se
traducen a la estructura nueva por el nombre del archivo:

- etiqueta → `datos_derivados/humming/etiquetas_<v>/<split>/<stem>.txt` (repo)
- imagen → `<frames>/humming_<v>/<split>/images/<stem>.jpg` (zona de trabajo, la regenera el notebook 01)

GT v1 = etiquetas de la versión 1 de Roboflow (S4, S5). GT v2 = versión corregida (S5, D-10).

In [ ]:
HUMMING = P["dd"] / "humming"
VV = P["dd"] / "vv"

def yolo_boxes(text, W, H):
    out = []
    for l in text.splitlines():
        if l.strip():
            _, cx, cy, w, h = map(float, l.split())
            out.append([(cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H])
    return out

def stem_de(ruta):
    return PureWindowsPath(ruta).stem

def label_path(ruta, version, split):
    return HUMMING / f"etiquetas_{version}" / split / f"{stem_de(ruta)}.txt"

def image_path(ruta, version, split):
    return P["frames"] / f"humming_{version}" / split / "images" / f"{stem_de(ruta)}.jpg"

def cargar_manifiesto(version):
    """split_manifest de la versión: ruta (original), split, origen (nuevo|vv), video_id, frame, split_roboflow, cajas, rf_file."""
    m = pd.read_csv(HUMMING / f"split_manifest_{version}.csv")
    m["etiqueta"] = [label_path(r.ruta, version, r.split) for r in m.itertuples()]
    m["imagen"] = [image_path(r.ruta, version, r.split) for r in m.itertuples()]
    return m

def cargar_gt_test(version, vids):
    """GT de los frames test_ en píxeles del video (load_test_gt de union_gt2.py y load_gt de tracking_test.py)."""
    sm = cargar_manifiesto(version)
    sm = sm[(sm.split == "test") & (sm.origen == "nuevo")]
    rows = []
    for r in sm.itertuples():
        v = vids[r.video_id]
        rows.append(dict(video_id=r.video_id, frame=int(r.frame), boxes=yolo_boxes(r.etiqueta.read_text(), v["W"], v["H"])))
    GT = pd.DataFrame(rows).sort_values(["video_id", "frame"]).reset_index(drop=True)
    GT["n_gt"] = GT.boxes.str.len()
    return GT

def cargar_gt_union(vids, version="v1"):
    """load_gt de union.py: vv (todas sus etiquetas de Roboflow menos ignorar_frames) + frames de train_/test_ del manifiesto."""
    import zipfile
    rows = []
    with zipfile.ZipFile(VV / "prueba_etiquetas_corregidas.zip") as z:
        ignore = set(z.read("ignorar_frames.txt").decode().split())
    sel = cargar_manifiesto(version)
    sel_vv = {(r.video_id, int(r.frame)): r.split for r in sel[sel.origen == "vv"].itertuples()}
    for lab in sorted((HUMMING / f"roboflow_{version}_vv").glob("vv*.txt")):
        stem = lab.stem.split("_jpg.rf.")[0]
        vid, f = stem.rsplit("_f", 1)
        f = int(f)
        if stem in ignore or vid not in vids:
            continue
        v = vids[vid]
        rows.append(dict(video_id=vid, frame=f, boxes=yolo_boxes(lab.read_text(), v["W"], v["H"]),
                         split=sel_vv.get((vid, f), "no_usado")))
    for r in sel[sel.origen == "nuevo"].itertuples():
        if r.video_id not in vids:
            continue
        v = vids[r.video_id]
        rows.append(dict(video_id=r.video_id, frame=int(r.frame), boxes=yolo_boxes(r.etiqueta.read_text(), v["W"], v["H"]),
                         split=r.split))
    GT = pd.DataFrame(rows).sort_values(["video_id", "frame"]).reset_index(drop=True)
    GT["n_gt"] = GT.boxes.str.len()
    return GT

## 4. Comparar con las cifras del sprint

En modo `artefactos` cada notebook recalcula sus tablas y las compara con las que se guardaron en el sprint
(`resultados/S*/...csv`). `comparar` une por las claves, mide la diferencia absoluta en cada columna numérica y
marca ✓ si todas quedan dentro de la tolerancia de `config.yaml`. **Nunca sobrescribe la referencia.**

In [ ]:
def comparar(nuevo: pd.DataFrame, referencia, claves, columnas=None, tol=None, nombre=""):
    ref = pd.read_csv(referencia) if not isinstance(referencia, pd.DataFrame) else referencia
    tol = TOL["metrica"] if tol is None else tol
    columnas = columnas or [c for c in ref.columns if c not in claves and pd.api.types.is_numeric_dtype(ref[c]) and c in nuevo]
    m = ref[claves + columnas].merge(nuevo[claves + columnas], on=claves, how="outer", suffixes=("_ref", "_nuevo"), indicator=True)
    falta = m[m._merge != "both"]
    d = pd.DataFrame({c: (m[f"{c}_nuevo"] - m[f"{c}_ref"]).abs() for c in columnas})
    m["max_dif"] = d.max(axis=1)
    ok = falta.empty and bool((m.max_dif.fillna(0) <= tol).all())
    print(f"{'✓' if ok else '✗'} {nombre}: {len(m)} filas, diferencia máxima {m.max_dif.max():.4g} (tolerancia {tol})"
          + (f", {len(falta)} filas sin pareja" if len(falta) else ""))
    if not ok:
        display(m[(m.max_dif > tol) | (m._merge != "both")].head(20))
    return ok

## 5. Tracking (S3, S5)

`track_video` corre `model.track(persist=True)` cuadro a cuadro con un modelo nuevo por video (estado limpio).
Configuraciones: `BASE` + `COMBO` sobre los YAML de Ultralytics, siempre con `gmc_method: none` (cámara fija).
`evaluate_tracks` empareja con la GT por Hungarian a IoU 0,5 y supone **un ave por video** (por eso `ids_por_ave`).
`build_trajectory` es la unión «actual» de S3 (30 frames, 1,5×); la que se usa ahora es `union_tuned` (sección 6).

In [ ]:
RUNS_TRACKING = RUNS / "tracking"
# Portado de video-val/resultados/tracking_v2/tracking_test.py
SHORT_S = 0.5  # tracks más cortos que esto se consideran sospechosos de ser cajas espurias
BASE = {"bytetrack": {"track_buffer": 60}, "ocsort": {"track_buffer": 60, "use_byte": True}}
COMBO = {"track_buffer": 300, "match_thresh": 0.95, "new_track_thresh": 0.5}

def configs_tracking():
    """Las 5 configuraciones de S5 (CONFIGS de tracking_test.py): nombre -> (pesos, tracker, overrides)."""
    NEW, OLD = DETECTOR, DETECTOR_ANTERIOR
    return {
        "a_nuevo_oc_combo": (NEW, "ocsort", {**BASE["ocsort"], **COMBO}),
        "b_nuevo_bt_combo": (NEW, "bytetrack", {**BASE["bytetrack"], **COMBO}),
        "c_nuevo_oc_base": (NEW, "ocsort", BASE["ocsort"]),
        "d_nuevo_bt_base": (NEW, "bytetrack", BASE["bytetrack"]),
        "e_viejo_oc_combo": (OLD, "ocsort", {**BASE["ocsort"], **COMBO}),   # solo referencia
    }

# Portado de video-val/resultados/tracking_v2/tracking_test.py
DECISION = ["a_nuevo_oc_combo", "b_nuevo_bt_combo", "c_nuevo_oc_base", "d_nuevo_bt_base"]  # e es solo referencia
CONFIGS = configs_tracking()

In [ ]:
def tracker_cfg(name: str, cfg_name: str, overrides: dict) -> Path:
    return escribir_tracker(name, overrides, RUNS_TRACKING / "cfg" / f"tracker_{cfg_name}.yaml")


In [ ]:
def evaluate_tracks(GT: pd.DataFrame, vid: str, tr: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    g = GT[GT.video_id == vid]
    by_frame = dict(tuple(tr.groupby("frame")))
    per = []
    for r in g.itertuples():
        p = by_frame.get(r.frame, tr.iloc[:0])
        gt = np.array(r.boxes, dtype=float).reshape(-1, 4)
        iou = iou_matrix(gt, to_xyxy(p))
        matched = []
        if iou.size:
            gi, pi = linear_sum_assignment(-iou)
            matched = [(i, j) for i, j in zip(gi, pi) if iou[i, j] >= IOU_MATCH]
        ids = p.track_id.to_numpy()
        per.append({"frame": r.frame, "n_gt": len(gt), "n_pred": len(p), "tp": len(matched),
                    "id": int(ids[matched[0][1]]) if matched else None,           # un ave por video
                    "iou": float(iou[matched[0]]) if matched else np.nan})
    per = pd.DataFrame(per)
    seq = per.id.dropna().astype(int).to_numpy()
    idsw = int((seq[1:] != seq[:-1]).sum()) if len(seq) else 0
    n_gt, n_pred, tp = per.n_gt.sum(), per.n_pred.sum(), per.tp.sum()
    idtp = int(pd.Series(seq).value_counts().iloc[0]) if len(seq) else 0
    m = {"video_id": vid, "tracker": tr.attrs["tracker"], "n_gt": int(n_gt), "TP": int(tp), "FN": int(n_gt - tp),
         "FP": int(n_pred - tp), "IDSW": idsw, "ids_por_ave": int(len(set(seq))),
         "ids_totales": int(tr.track_id.nunique()),
         "recall": tp / max(n_gt, 1), "precision": tp / max(n_pred, 1),
         "MOTA": 1 - (n_gt - tp + n_pred - tp + idsw) / max(n_gt, 1),
         "IDF1": 2 * idtp / max(n_gt + n_pred, 1), "IoU_medio": per.iou.mean(),
         "fps_proceso": tr.attrs.get("fps_proceso"), "idtp": idtp}
    return m, per

In [ ]:
# Portado de video-val/resultados/tracking_v2/tracking_test.py
INTERP_MAX_GAP = 10
JOIN_MAX_GAP = 30
JOIN_MAX_DIST = 1.5
MIN_FRAG_LEN = 15

# Portado de video-val/resultados/tracking_v2/tracking_test.py
def build_trajectory(tr: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    fps = tr.attrs["fps_declared"]
    if tr.empty:
        return tr.iloc[:0], {"ids_unidos": [], "ids_descartados": []}
    span = tr.groupby("track_id").frame.agg(["min", "max", "size"]).sort_values("size", ascending=False)
    main = [span.index[0]]
    discarded = []
    for tid in span.index[1:]:
        s = span.loc[tid]
        used = tr[tr.track_id.isin(main)]
        if ((used.frame >= s["min"]) & (used.frame <= s["max"])).any():
            discarded.append(int(tid)); continue
        before, after = used[used.frame < s["min"]], used[used.frame > s["max"]]
        frag = tr[tr.track_id == tid].sort_values("frame")
        ok, near = False, False
        for ref, ends, gap in ((before.sort_values("frame").tail(1), frag.head(1), s["min"] - before.frame.max() if len(before) else np.inf),
                               (after.sort_values("frame").head(1), frag.tail(1), after.frame.min() - s["max"] if len(after) else np.inf)):
            if len(ref) and gap <= JOIN_MAX_GAP:
                near = True
                size = (ref.w_px.iloc[0] + ref.h_px.iloc[0]) / 2
                d = np.hypot(ref.cx_px.iloc[0] - ends.cx_px.iloc[0], ref.cy_px.iloc[0] - ends.cy_px.iloc[0])
                ok |= d <= JOIN_MAX_DIST * size
        if not near:
            ok = s["size"] >= MIN_FRAG_LEN
        (main if ok else discarded).append(int(tid))
    t = tr[tr.track_id.isin(main)].sort_values("frame").drop_duplicates("frame")
    t = t.set_index("frame")[["cx_px", "cy_px", "w_px", "h_px", "conf"]]
    full = t.reindex(range(t.index.min(), t.index.max() + 1))
    missing = full.cx_px.isna()
    run_id = (~missing).cumsum()
    run_len = missing.groupby(run_id).transform("sum")
    fill = missing & (run_len <= INTERP_MAX_GAP)
    interp = full.interpolate(limit_area="inside")
    full.loc[fill, ["cx_px", "cy_px", "w_px", "h_px"]] = interp.loc[fill, ["cx_px", "cy_px", "w_px", "h_px"]]
    full["interpolado"] = fill
    full = full[full.cx_px.notna()].copy()
    full["segmento"] = (np.diff(full.index, prepend=full.index[0]) > 1).cumsum()
    full["t_s"] = full.index / fps
    v = np.full(len(full), np.nan)
    for _, seg in full.groupby("segmento"):
        if len(seg) >= 7:
            w = min(11, len(seg) // 2 * 2 - 1)
            vx = savgol_filter(seg.cx_px, w, 2, deriv=1) * fps
            vy = savgol_filter(seg.cy_px, w, 2, deriv=1) * fps
            v[full.index.get_indexer(seg.index)] = np.hypot(vx, vy)
    full["v_px_s"] = v
    full = full.reset_index().rename(columns={"index": "frame"})
    return full[["frame", "t_s", "cx_px", "cy_px", "w_px", "h_px", "conf", "interpolado", "segmento", "v_px_s"]], \
        {"ids_unidos": [int(x) for x in main], "ids_descartados": discarded}


def short_tracks(tr: pd.DataFrame, GT_v: pd.DataFrame) -> pd.DataFrame:
    fps = tr.attrs["fps_declared"]
    g = tr.groupby("track_id").agg(ini=("frame", "min"), fin=("frame", "max"), n=("frame", "size"),
                                   cx=("cx_px", "mean"), cy=("cy_px", "mean"), w=("w_px", "mean"),
                                   h=("h_px", "mean"), conf=("conf", "mean"))
    g["dur_s"] = (g.fin - g.ini + 1) / fps
    g = g[g.dur_s < SHORT_S].reset_index()
    # ¿había ave etiquetada en el frame GT más cercano?
    gf = GT_v.frame.to_numpy()
    g["ave_en_gt_cercano"] = [bool(GT_v.n_gt.iloc[np.abs(gf - (a + b) / 2).argmin()]) if len(gf) else None
                              for a, b in zip(g.ini, g.fin)]
    g["t_ini_s"] = g.ini / fps
    return g


def draw_traj(video: Path, tr, traj, GT_v, title, out):
    cap = cv2.VideoCapture(str(video))
    mid = int(traj.frame.median()) if len(traj) else 0
    cap.set(cv2.CAP_PROP_POS_FRAMES, mid); ok, f = cap.read(); cap.release()
    fig, ax = plt.subplots(figsize=(12, 12 * f.shape[0] / f.shape[1]))
    ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB), alpha=0.55)
    for tid, g in tr.groupby("track_id"):           # todos los IDs, finos
        ax.plot(g.cx_px, g.cy_px, lw=0.8, alpha=0.7)
    sc = None
    for _, seg in traj.groupby("segmento"):         # trayectoria final, gruesa y coloreada por tiempo
        sc = ax.scatter(seg.cx_px, seg.cy_px, c=seg.t_s, cmap="viridis", s=8, vmin=0, vmax=tr.attrs["n_frames"] / tr.attrs["fps_declared"])
    gx = [(b[0][0] + b[0][2]) / 2 for b in GT_v.boxes if b]; gy = [(b[0][1] + b[0][3]) / 2 for b in GT_v.boxes if b]
    ax.scatter(gx, gy, marker="x", c="red", s=22, label="verdad de terreno")
    ax.set_title(title, fontsize=10); ax.axis("off"); ax.legend(loc="lower left", fontsize=8)
    if sc is not None:
        fig.colorbar(sc, ax=ax, shrink=0.6, label="t (s)")
    fig.savefig(out, dpi=110, bbox_inches="tight"); plt.close(fig)

`get_tracks` es la única puerta a las trayectorias detalladas (`__detallada.csv`, una fila por caja y frame).
**Regla de S3: la trayectoria detallada nunca se sobrescribe.** Orden de búsqueda:

1. `datos_derivados/tracks/<config>/` — las congeladas en el repo (las de S5, con las que se tomaron D-09 y D-10);
2. `runs/tracking/<config>/` — las que haya calculado este equipo;
3. si no existe ninguna, se calcula con `track_video` y se guarda en 2.

En modo `recalcular` se salta 1, para comprobar que el tracking se reproduce.

In [ ]:
def get_tracks(vid, v, cfg_name=None):
    cfg_name = cfg_name or CFG["vigente"]["tracker"]
    weights, tracker, over = CONFIGS[cfg_name]
    dirs = ([P["dd"] / "tracks" / cfg_name] if MODO == "artefactos" else []) + [RUNS_TRACKING / cfg_name]
    for d in dirs:
        out = d / f"{vid}__detallada.csv"
        if out.exists():
            tr = pd.read_csv(out); tr.attrs.update(json.loads(out.with_suffix(".meta.json").read_text(encoding="utf-8")))
            return tr
    out = dirs[-1] / f"{vid}__detallada.csv"; out.parent.mkdir(parents=True, exist_ok=True)
    print("tracking", cfg_name, vid, flush=True)
    tr = track_video(weights, v["path"], tracker, tracker_cfg(tracker, cfg_name, over))
    tr.to_csv(out, index=False); out.with_suffix(".meta.json").write_text(json.dumps(tr.attrs, indent=2), encoding="utf-8")
    return tr

## 6. Unión de fragmentos a nivel de visita (S5)

`union_tuned(tr, max_gap_s, max_dist, cuts)`: recorre las filas aceptadas en orden temporal; dos filas consecutivas
siguen en el mismo segmento si el hueco es ≤ `max_gap_s`, la distancia ≤ `max_dist` × lado medio de la caja y no hay
corte de escena entre ellas. Los huecos dentro de un segmento se interpolan (**puente**). Vigente (D-10): 1 s y 2,5×.

`visit_metrics`: una **visita** es una racha de frames GT con ave. `IDF1_visita` empareja visitas con segmentos;
un **puente falso** es un segmento que cubre dos visitas. `bridges` cuenta los puentes **largos** (> ¼ del lado corto).

In [ ]:
_scene_cuts_colibri = scene_cuts
def scene_cuts(path: Path) -> list:
    return _scene_cuts_colibri(path, RUNS / "union" / "cortes")


In [ ]:
def union_reference(tr):
    traj, _ = build_trajectory(tr)
    return traj.rename(columns={"segmento": "segment_id"})

In [ ]:
# Portado de video-val/resultados/tracking_v2/union/union.py
def visit_metrics(traj, g):
    """g: frames GT de un video. Frames con 2+ cajas se excluyen (no forman ni separan visitas)."""
    g = g[g.n_gt <= 1].sort_values("frame")
    present = (g.n_gt == 1).to_numpy()
    visit = np.where(present, np.cumsum(np.r_[present[0], present[1:] & ~present[:-1]]) - 1, -1)
    by_frame = traj.set_index("frame") if len(traj) else None
    match, n_pred = [], 0  # (visita, segmento) por caja GT cubierta
    for (f, boxes), vi in zip(zip(g.frame, g.boxes), visit):
        if by_frame is None or f not in by_frame.index:
            continue
        r = by_frame.loc[f]
        n_pred += 1
        if vi >= 0:
            pb = np.array([[r.cx_px - r.w_px / 2, r.cy_px - r.h_px / 2, r.cx_px + r.w_px / 2, r.cy_px + r.h_px / 2]])
            if iou_matrix(np.array(boxes), pb)[0, 0] >= IOU_MATCH:
                match.append((vi, int(r.segment_id)))
    n_vis, n_box = int(visit.max() + 1) if len(visit) else 0, int(present.sum())
    M = pd.DataFrame(match, columns=["v", "s"])
    seg_per_visit = M.groupby("v").s.nunique().reindex(range(n_vis), fill_value=0)
    bridges = int((M.groupby("s").v.nunique() - 1).clip(lower=0).sum()) if len(M) else 0
    if len(M):
        C = M.value_counts().unstack(fill_value=0)
        ri, ci = linear_sum_assignment(-C.to_numpy())
        idtp = int(C.to_numpy()[ri, ci].sum())
    else:
        idtp = 0
    return dict(visitas=n_vis, cajas=n_box, cubiertas=len(M), pred_en_gt=n_pred, idtp=idtp,
                seg_por_visita=float(seg_per_visit[seg_per_visit > 0].mean()) if (seg_per_visit > 0).any() else 0.0,
                visitas_sin_segmento=int((seg_per_visit == 0).sum()), puentes_falsos=bridges,
                cobertura=len(M) / max(n_box, 1), IDF1_visita=2 * idtp / max(n_box + n_pred, 1),
                segmentos=int(traj.segment_id.nunique()) if len(traj) else 0)

def aggregate(rows):
    d = pd.DataFrame(rows)
    s = d[["visitas", "cajas", "cubiertas", "pred_en_gt", "idtp", "visitas_sin_segmento", "puentes_falsos", "segmentos"]].sum()
    w = d.visitas - d.visitas_sin_segmento
    return dict(**s.to_dict(), seg_por_visita=float((d.seg_por_visita * w).sum() / max(w.sum(), 1)),
                cobertura=s.cubiertas / max(s.cajas, 1), IDF1_visita=2 * s.idtp / max(s.cajas + s.pred_en_gt, 1))

def union_vigente(tr, cuts):
    u = CFG["vigente"]["union"]
    traj = union_tuned(tr, u["max_gap_s"], u["max_dist"], cuts).copy()
    traj["es_puente"] = traj.interpolado.astype(bool)
    return traj

## 7. Siete puntos (S7)

Suavizado Savitzky-Golay por segmento (~0,2 s, grado 2); 7 puntos **en total** por video (D-12) con dos métodos:
uniforme en el tiempo con ave y salientes (Visvalingam-Whyatt). Un punto que cae en un puente se mueve al observado
más cercano. Por los puntos de cada tramo pasa una Catmull-Rom centrípeta; los huecos entre segmentos no se dibujan.

In [ ]:
# Portado de video-val/resultados/trayectorias_7pts/trayectorias_7pts.py
NAMES = {"doj1xa6ksqm": "DoJ1", "gu30nofexla": "Gu30", "i5befvb5mxg": "i5bE", "j4pkgwgjaus": "J4PK"}

# Cuerpo por video de main() de trayectorias_7pts.py, separado para usarlo también en la herramienta
siete_puntos = puntos_clave


## 8. Profundidad (S8)

Todos los modelos de profundidad entran por la misma interfaz: `map(frame_idx, frame, flip, secondary)` devuelve un
mapa alineado con el frame original. `yolo26` es profundidad (mayor = más lejos); `vda` es **disparidad** (mayor = más
cerca) y lee los mapas que calculó el notebook 05. `z_norm = (z_ave − z_flor)/(z_fondo − z_flor)`: 0 = flor,
1 = fondo, negativo = delante, en los dos dominios (D-14). La magnitud no es comparable entre dominios.

In [ ]:
# read_frame, motion_mask, make_backend y las demás funciones de lectura vienen de colibri.profundidad


In [ ]:
_mapas_vda_colibri = mapas_vda
def mapas_vda(videos_vda, cache, sizes=None):
    return _mapas_vda_colibri(videos_vda, cache, sizes or CFG["vda"]["sizes"], ZONA / CFG["vda"]["repo"], ZONA / CFG["vda"]["checkpoint"])


In [ ]:
print("00_comun listo:", len(FUENTES), "videos en fuentes.csv |", "detector vigente", DETECTOR.relative_to(REPO),
      "existe" if DETECTOR.exists() else "FALTA")